Since the last model was not accurate due to a non correlating dataset, let's try building a new one with the help of a new dataset

In [2]:
import pandas as pd
df=pd.read_excel("/content/Perfomance.xls")
df.shape

(1200, 28)

In [3]:
df.columns.tolist()

['EmpNumber',
 'Age',
 'Gender',
 'EducationBackground',
 'MaritalStatus',
 'EmpDepartment',
 'EmpJobRole',
 'BusinessTravelFrequency',
 'DistanceFromHome',
 'EmpEducationLevel',
 'EmpEnvironmentSatisfaction',
 'EmpHourlyRate',
 'EmpJobInvolvement',
 'EmpJobLevel',
 'EmpJobSatisfaction',
 'NumCompaniesWorked',
 'OverTime',
 'EmpLastSalaryHikePercent',
 'EmpRelationshipSatisfaction',
 'TotalWorkExperienceInYears',
 'TrainingTimesLastYear',
 'EmpWorkLifeBalance',
 'ExperienceYearsAtThisCompany',
 'ExperienceYearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager',
 'Attrition',
 'PerformanceRating']

Let's first check for the correlation before moving further

In [4]:
df_corr = df.copy()
df_corr = pd.get_dummies(df_corr)
print(df_corr.corr()['PerformanceRating'].sort_values(ascending=False).round(4).head(15))

PerformanceRating             1.0000
EmpEnvironmentSatisfaction    0.3956
EmpLastSalaryHikePercent      0.3337
EmpDepartment_Development     0.1740
EmpJobRole_Developer          0.1503
EmpWorkLifeBalance            0.1244
EmpNumber_E100958             0.0586
EmpNumber_E100983             0.0586
EmpNumber_E100788             0.0586
EmpNumber_E100792             0.0586
EmpNumber_E100992             0.0586
EmpNumber_E100987             0.0586
EmpNumber_E100985             0.0586
EmpNumber_E100799             0.0586
EmpNumber_E100803             0.0586
Name: PerformanceRating, dtype: float64


Employee's salary hike might cause leakage risk, so let's keep other important columns

In [6]:
keep_cols = [
    'EmpEnvironmentSatisfaction',    # 0.39 — strongest signal
    'EmpWorkLifeBalance',            # 0.12 — decent signal
    'EmpJobInvolvement',             # work behavior
    'EmpJobSatisfaction',            # engagement
    'EmpRelationshipSatisfaction',   # work behavior
    'TrainingTimesLastYear',         # growth effort
    'YearsSinceLastPromotion',       # achievement
    'ExperienceYearsInCurrentRole',  # role mastery
    'TotalWorkExperienceInYears',    # overall experience
    'OverTime',                      # dedication
    'PerformanceRating'              # target
]

df = df[keep_cols]
print(df.shape)
print(df.columns.tolist())

(1200, 11)
['EmpEnvironmentSatisfaction', 'EmpWorkLifeBalance', 'EmpJobInvolvement', 'EmpJobSatisfaction', 'EmpRelationshipSatisfaction', 'TrainingTimesLastYear', 'YearsSinceLastPromotion', 'ExperienceYearsInCurrentRole', 'TotalWorkExperienceInYears', 'OverTime', 'PerformanceRating']


Okay let's check if there are missing values, what's the value distribution and also datatypes it has to make sure we convert the strings to float or int before proceeding on developing the model

In [7]:
print(df.isnull().sum())
print(df["PerformanceRating"].value_counts())
print(df["PerformanceRating"].value_counts(normalize=True).round(3)*100)
print(df.dtypes)

EmpEnvironmentSatisfaction      0
EmpWorkLifeBalance              0
EmpJobInvolvement               0
EmpJobSatisfaction              0
EmpRelationshipSatisfaction     0
TrainingTimesLastYear           0
YearsSinceLastPromotion         0
ExperienceYearsInCurrentRole    0
TotalWorkExperienceInYears      0
OverTime                        0
PerformanceRating               0
dtype: int64
PerformanceRating
3    874
2    194
4    132
Name: count, dtype: int64
PerformanceRating
3    72.8
2    16.2
4    11.0
Name: proportion, dtype: float64
EmpEnvironmentSatisfaction       int64
EmpWorkLifeBalance               int64
EmpJobInvolvement                int64
EmpJobSatisfaction               int64
EmpRelationshipSatisfaction      int64
TrainingTimesLastYear            int64
YearsSinceLastPromotion          int64
ExperienceYearsInCurrentRole     int64
TotalWorkExperienceInYears       int64
OverTime                        object
PerformanceRating                int64
dtype: object


Okay now we have one serious issue and one small factor to focus on

The serious issue is that Rating 3 is dominating , so the model might be too biased towards the rating 3, so we have to add some data for rating 2 and 4 too, Let's use synthetic data which would actually be linearly related to the actual data

In [8]:
df["OverTime"].value_counts()

,count
OverTime,
No,847
Yes,353


The OverTime column has "Yes" or "No" so we should map it as 0 and 1

In [10]:
df["OverTime"]=df["OverTime"].map({"Yes":1,"No":0})
df["OverTime"].value_counts()

,count
OverTime,
0,847
1,353


Secondly let's handle the issue of non dominant rating data

In [13]:
from imblearn.over_sampling import SMOTE
X=df.drop(["PerformanceRating"],axis=1)
y=df["PerformanceRating"]
smote=SMOTE()
X_resampled,y_resampled=smote.fit_resample(X,y)
print(pd.Series(y_resampled).value_counts())
print(pd.Series(y_resampled).value_counts(normalize=True).round(3)*100)

PerformanceRating
3    874
4    874
2    874
Name: count, dtype: int64
PerformanceRating
3    33.3
4    33.3
2    33.3
Name: proportion, dtype: float64


Okay now the dataset is evenly splitted

Now let's start splitting the dataset into 80% training and 20% testing

In [15]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X_resampled,y_resampled
                                               ,test_size=0.2,
                                               random_state=42,
                                               stratify=y_resampled)
print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")

X_train: (2097, 10)
X_test : (525, 10)


Now let's make sure that the values are evenly distributed

In [16]:
print("\nTrain distribution:")
print(pd.Series(y_train).value_counts(normalize=True).round(3) * 100)
print("\nTest distribution:")
print(pd.Series(y_test).value_counts(normalize=True).round(3) * 100)


Train distribution:
PerformanceRating
3    33.3
4    33.3
2    33.3
Name: proportion, dtype: float64

Test distribution:
PerformanceRating
4    33.3
2    33.3
3    33.3
Name: proportion, dtype: float64


Hence it is verified

Now we are set to build the RandomForestClassifier model

In [17]:
from sklearn.ensemble import RandomForestClassifier
rf_model=RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train,y_train)

RandomForestClassifier(max_depth=10, min_samples_leaf=4, min_samples_split=10,
                       n_jobs=-1, random_state=42)

Training is done, let's evaluate the model now

In [18]:
from sklearn.metrics import classification_report,confusion_matrix
y_pred=rf_model.predict(X_test)
print(classification_report(y_test,y_pred))
print("Confusion Matrix")
print(confusion_matrix(y_test,y_pred))

              precision    recall  f1-score   support

           2       0.90      0.98      0.94       175
           3       0.86      0.63      0.73       175
           4       0.78      0.91      0.84       175

    accuracy                           0.84       525
   macro avg       0.85      0.84      0.84       525
weighted avg       0.85      0.84      0.84       525

Confusion Matrix
[[172   3   0]
 [ 19 111  45]
 [  1  15 159]]


We have got ourselves a pretty good model with pretty good accuracy

In [20]:
feat_imp = pd.DataFrame({
    'Feature'    : X_train.columns,
    'Importance' : rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(feat_imp)

                        Feature  Importance
0    EmpEnvironmentSatisfaction    0.391039
6       YearsSinceLastPromotion    0.190749
7  ExperienceYearsInCurrentRole    0.106839
8    TotalWorkExperienceInYears    0.065535
9                      OverTime    0.049824
4   EmpRelationshipSatisfaction    0.041174
2             EmpJobInvolvement    0.040345
1            EmpWorkLifeBalance    0.039942
5         TrainingTimesLastYear    0.038634
3            EmpJobSatisfaction    0.035920


It is seen the that the employee environment satisfaction plays a major role , so we have to make sure that the given input to this model should be not biased as it determines the performance rating

My suggestion would be to take weighted average from all sorts of review about the employee rather than sticking to simple rating system